# 01: Data Preprocessing & Masking
**Description:** This notebook handles the merging of NDWI TIF files, creation of water masks, and generation of elevation-based statistics.
---
**Note:** Ensure your data is placed in the `../data/raw/` and `../data/processed/` folders relative to this notebook.

In [2]:
#--- IMPORT LIBRARIES --------------------


import os #so python can interact directly with computer
import matplotlib.pyplot as plt #for plotting
import matplotlib.colors as mcolors #colors
from matplotlib.colors import LightSource #for creating hillshade
from matplotlib.colors import ListedColormap #for coloring in plots
from scipy.stats import linregress #compute linear regression line for statistics in plots
from scipy.stats import theilslopes #compute Theil-Sen slope for statistics in plots
import numpy as np #for arrays
import xarray as xr #for easy access to netCDF files
import pandas as pd #for tabulated data and also handles times nicely and shapefiles
import rioxarray as rxr #for multi-dimensional raster data
import rasterio #for visualizing rasters
from rasterio.mask import mask #for clipping or masking in rasters
from rasterio.transform import array_bounds #for setting bounds
from rioxarray.merge import merge_datasets, merge_arrays #for merging rasters
import geopandas as gpd #for reading and managing vector files
import re #for loops
from shapely.geometry import mapping #for mapping with coordinates

In [ ]:
#--- MERGE NDWI TIFS & SAVE --------------------


# Settings
years = [1988, 1991, 1994, 1997, 2000, 2003, 2006, 2009, 2012, 2015, 2018, 2021, 2024] #loop through all years
output_folder = "Full_NDWIg" 
os.makedirs(output_folder, exist_ok=True)

# Loop through all years creating full NDWIg TIF files and merge
for year in years:
    # Load data and select NDWIg band 1
    path137 = rxr.open_rasterio(f"NDWIg_137-41/NDWIg_137_041_y{year}.tif").sel(band=1).squeeze() #1=NDWIg
    path138 = rxr.open_rasterio(f"NDWIg_138-41/NDWIg_138_041_y{year}.tif").sel(band=1).squeeze()
    
    # Merge the two selected NDWIg layers of each tile
    merged_ndwi = merge_arrays([path137, path138])

    # Fix metadata: delete 'long_name' (we only 1 band is selected) and set new name
    if 'long_name' in merged_ndwi.attrs:
        del merged_ndwi.attrs['long_name']
    merged_ndwi.attrs['long_name'] = 'NDWIg'
    
    # Save as georeferenced TIF
    out_name = f"{output_folder}/ndwig_merged_{year}.tif"
    merged_ndwi.rio.to_raster(out_name, dtype="float32") #float32 saves space, half of float64

In [ ]:
#--- CREATE FULL WATER MASKS FROM NDWI TIFS --------------------


# Settings
input_folder = "Full_NDWIg"
output_folder = "Full_Masks"
os.makedirs(output_folder, exist_ok=True)

# List of years
years = [1988, 1991, 1994, 1997, 2000, 2003, 2006, 2009, 2012, 2015, 2018, 2021, 2024]

# Water threshold
threshold = 0.3

for year in years:
    # load the merged NDWI file
    file_path = f"{input_folder}/ndwig_merged_{year}.tif"
    
    if os.path.exists(file_path):
        print(f"Processing year: {year}")
        ndwi = rxr.open_rasterio(file_path).squeeze()

        # apply threshold to create binary mask
        water_mask = xr.where(ndwi > threshold, 1, 0).astype("uint8") # water = 1, land = 0

        # preserve spatial metadata (CRS and Transform)
        water_mask.rio.write_crs(ndwi.rio.crs, inplace=True)
        
        # save as TIF
        out_name = f"{output_folder}/bhutan_full_mask_{year}.tif"
        water_mask.rio.to_raster(out_name)
    else:
        print(f"Warning: {file_path} not found.")

In [ ]:
#--- STACK COMPOSITES: ndwig --------------------


# Settings
folder = "Full_NDWIg/"
years = [1988, 1991, 1994, 1997, 2000, 2003, 2006, 2009, 2012, 2015, 2018, 2021, 2024]

mask_list = []

# Loop
for year in years:
    file_path = f"{folder}ndwig_merged_{year}.tif" #open file
    # drop the varibale 'band' and replace with 'year', so every composite in stack has the appropriate year information
    da = xr.open_dataarray(file_path).squeeze().drop_vars('band', errors='ignore')
    da = da.expand_dims(year=[year])
    mask_list.append(da)

# Create a 3D stack (year: 13, y: ..., x: ...)
ndwig_stack = xr.concat(mask_list, dim='year')

In [ ]:
#--- STACK COMPOSITES: water masks --------------------


# Settings
folder = "Full_Masks/"
years = [1988, 1991, 1994, 1997, 2000, 2003, 2006, 2009, 2012, 2015, 2018, 2021, 2024]

mask_list = []

for year in years:
    file_path = f"{folder}bhutan_full_mask_{year}.tif" #open file
    # drop the varibale 'band' and replace with 'year', so every composite in stack has the appropriate year information
    da = xr.open_dataarray(file_path).squeeze().drop_vars('band', errors='ignore')
    da = da.expand_dims(year=[year]) #every year (1988, 1994...) becomes a band (1, 2, 3...)
    mask_list.append(da)

# Create a 3D stack (year: 13, y: ..., x: ...)
watermask_stack = xr.concat(mask_list, dim='year')

# Sum across time dimension to get frequency
frequency = watermask_stack.sum(dim='year')

In [ ]:
#--- Export stacks --------------------


# Save stack as Multi-Band TIFF
frequency.rio.to_raster("Watermask_Sum_StackClean.tif")

In [ ]:
#--- Reomve 1991 from created stacks --------------------


ndwig_stack = ndwig_stack.drop_sel(year=1991)

watermask_stack = watermask_stack.drop_sel(year=1991)

frequency = watermask_stack.sum(dim='year')

In [ ]:
#--- CLIP WATER MASK STACK TO BHUTAN BOUNDARY --------------------

# Read in data
watermask_stack = xr.open_dataarray('../data/raw/Watermask_Timeseries_StackClean.tif')

# load boundary outline shape and convert to same CRS
bhutan_boundary = gpd.read_file('../data/raw/bhutan_boundary/bhutan_boundary.shp').to_crs(watermask_stack.rio.crs)

# Clipping
watermask_stack_bhutan = watermask_stack.rio.clip(bhutan_boundary.geometry, bhutan_boundary.crs, all_touched=True) #erst auf Bhutan zuschneiden

# Export
watermask_stack_bhutan.astype('float32').rio.to_raster("Final_Watermask_Timeseries.tif")

In [ ]:
#--- COUNT WATER PIXELS IN MULTI-BAND STACK (FINAL MASK) --------------------


# Load data
stack = rxr.open_rasterio('../data/raw/Final_Watermask_Timeseries.tif')

# Prepare year array
years = [1988, 1994, 1997, 2000, 2003, 2006, 2009, 2012, 2015, 2018, 2021, 2024]

final_results = []

# Loop through bands (each band is a year)
for i, year in enumerate(years):
    # Select band
    band_data = stack.isel(band=i)
    
    # Count pixels (which are 1's)
    count = int((band_data == 1).sum())
    
    final_results.append({
        'year': year,
        'water_pixel_count': count,
        'area_km2': count * 0.0009 #add column of computed area, count to km^2 (conversion: 900m² / 1'000'000)
    })

# Export as CSV
final_water = pd.DataFrame(final_results)
final_water.to_csv("final_water.csv", index=False)

In [ ]:
#--- THEIL-SEN: pro pixel trend --------------------


# Load data
stack = xr.open_dataarray('Final_NDWIg_Timeseries_StackClean.tif').squeeze() #make it 2D

# define years, excluding year 1991!
years = [1988, 1994, 1997, 2000, 2003, 2006, 2009, 2012, 2015, 2018, 2021, 2024] #1991 is excluded
stack = stack.assign_coords(band=years).rename({'band': 'year'}) #assign years to band number

# Function for per pixel trend
def calculate_slope(y):
    if np.isnan(y).all():
        return np.nan
    res = theilslopes(y, years) #compute trend
    return res[0]

# Mask (Turbo-Boost)
#frequency = xr.open_dataarray('../data/raw/Watermask_Sum_StackClean.tif').squeeze() #use frequency map, so only areas with permanent water
#small_stack = stack_clean.where(frequency > 0)

# Apply Trend
ndwig_trend = xr.apply_ufunc(
    calculate_slope,
    stack,
    input_core_dims=[['year']],
    vectorize=True,
    output_dtypes=[float])

In [ ]:
#--- Export Theil-Sen Slope Trend as GeoTIFF --------------------

ndwig_trend.rio.to_raster(
    "NDWIg_TeilSen_Trend_PerPixel_Clean.tif",
    compress="LZW",      #saving space
    tiled=True, 
    dtype="float32"      #important, as trends have many decimals
)

In [ ]:
#--- CREATE HILLSHADE --------------------


# Load DEM
dem_xr = rxr.open_rasterio('DEM_Bhutan.tif') #with rioxarray to keep it as an xarray (and therefore crs)

# Clean and rescale DEM
dem = (dem_xr / 100.0).where(dem_xr > -1000)
print(f"Corrected elevation range: {dem.min().values:.2f} to {dem.max().values:.2f} meters") #check

# Prepare DEM for hillshade
dem_2d = dem.squeeze().values #remove "band"-dimension (squeeze) and convert to Numpy-Array (.values)

# Create Hillshade
ls = plt.matplotlib.colors.LightSource(azdeg=315, altdeg=45) #azdeg=315 (Sun from NW) is the standard for mountain maps, higher sun to make it brighter
hillshade = ls.hillshade(dem_2d, vert_exag=1.5)

# Export 
# Put hillshade back into an xarray DataArray
hillshade_xr = xr.DataArray(hillshade, coords=dem.squeeze().coords, dims=dem.squeeze().dims) #using the coordinates from your original DEM, make it 2d

# Clear attributes that might cause export errors
hillshade_xr.attrs = {}

# Export as a 2D GeoTIFF
hillshade_xr.rio.to_raster("hillshade_xr.tif")

In [ ]:
#--- FUNCTION: for getting elevation stats --------------------


def get_yearly_elevation_stats(final_stack, dem_layer, bin_size=50):

    """"
    Input ----------------------------------------------------------------
    final_stack: water mask stack (TIF)
    dem_layer: DEM (TIF)
    bin_size: 50 (default)
    Output ---------------------------------------------------------------
    df_result: Every water area per elevation bin for every composite year
    """

    results = {}
    
    # Define bins
    bins = np.arange(3000, 6550, bin_size)
    
    # Use of created dimension 'year' (instead of 'band')
    n_bands = final_stack.sizes['year']
    
    # Get year number from 'year' coordinate
    if 'year' in final_stack.coords:
        band_names = final_stack['year'].values
    else:
        # Fallback, in case dimension has diffferent name
        band_names = [f"year_{i}" for i in range(n_bands)]
        
    for i in range(n_bands):
        # Choose year
        band_mask = final_stack.isel(year=i)
        
        # Extract elevation value
        h = dem_layer.where(band_mask == 1).values.flatten()
        h = h[~np.isnan(h)]
        
        # Compute histogramm
        counts, _ = np.histogram(h, bins=bins)
        
        # Convert to km^2
        area_km2 = counts * 0.0009 
        
        # Handle column name
        col_name = str(band_names[i])
        results[col_name] = area_km2
        
    df_result = pd.DataFrame(results, index=bins[:-1])
    return df_result

In [ ]:
#--- GET STATISTICS OF ELEVATION AND WATER FREQUENCY: with function get_yearly_elevation_stats() --------------------

# Load data
final_stack = rxr.open_rasterio('Final_Watermask_TimeseriesClean.tif')
years_clean = [1988, 1994, 1997, 2000, 2003, 2006, 2009, 2012, 2015, 2018, 2021, 2024] #define years, exclude 1991
final_stack = final_stack.assign_coords(band=years_clean).rename({'band': 'year'}) #assign years to band number

dem_xr = rxr.open_rasterio('DEM_Bhutan.tif')
dem = (dem_xr / 100.0).where(dem_xr > -1000) #clean and rescale DEM

freq = xr.open_dataarray('Final_Frequency.tif')

# Alignement of DEM with frequency data
dem_matched = dem.rio.reproject_match(freq)

# Apply function
df_yearly_elevation = get_yearly_elevation_stats(final_stack, dem_matched)

# Create header for elevation values column
df_export = df_yearly_elevation.reset_index()
df_export = df_export.rename(columns={'index': 'elevation_m'})

# Export as CSV
df_export.to_csv('elevation_stats.csv', index=False)